In [1]:
import os
import json
import traceback
from bbtransformer import tune_hyperparameters, prepare_fmri_data


def run_tuning_workflow(
    target_name: str,
    data_path: str,
    pheno_path: str,
    tuning_config: dict,
    n_trials: int = 30,
    base_batch: int = 64,
    random_seed: int = 42,
    weights_dir: str = 'weights',
    results_dir: str = 'results'
):
    """
    Executes a complete hyperparameter tuning workflow for fMRI-based classification.

    Parameters:
        target_name (str): Name of the clinical target (e.g., 'ASD', 'Parkinsons').
        data_path (str): Path to the .npz fMRI data file.
        pheno_path (str): Path to the phenotype CSV file.
        tuning_config (dict): Configuration dictionary for hyperparameter search space.
        n_trials (int): Number of Optuna trials to run.
        base_batch (int): Initial batch size; will reduce on OOM.
        random_seed (int): Random seed for reproducibility.
        weights_dir (str): Directory to save best hyperparameters.
        results_dir (str): Directory for logging/results (created but not used here).

    Returns:
        tuple: (best_hyperparameters_dict or None, optuna_study or None)
    """
    # Ensure output directories exist
    os.makedirs(weights_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)

    def load_data_with_fallback():
        """Attempts to load data with decreasing batch sizes on OOM risk."""
        for batch_size in [base_batch, 32, 16]:
            try:
                print(f"   Attempting batch_size = {batch_size}...")
                loaders = prepare_fmri_data(
                    data_path=data_path,
                    pheno_path=pheno_path,
                    target_column=target_name,
                    batch_size=batch_size,
                    random_seed=random_seed
                )
                print(f"   ✅ Success with batch_size = {batch_size}")
                return loaders, batch_size
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"   ❌ OOM with batch_size = {batch_size}, trying smaller...")
                    continue
                else:
                    raise e
        raise RuntimeError("All batch sizes failed due to OOM.")

    try:
        print(f"🚀 Starting hyperparameter tuning for '{target_name}'")
        print(f"📂 Data: {data_path}")

        (train_loader, val_loader, _, metadata), used_batch = load_data_with_fallback()
        feature_dim = metadata['feature_dim']

        print(f"\n✅ Data loaded:")
        print(f"   - Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}")
        print(f"   - Feature dim: {feature_dim} | Batch size: {used_batch}")

        best_hp, study = tune_hyperparameters(
            train_loader=train_loader,
            val_loader=val_loader,
            feature_dim=feature_dim,
            n_trials=n_trials,
            search_config=tuning_config,
            target_name=target_name
        )

        if best_hp is not None:
            print(f"\n✅ Tuning succeeded!")
            print(f"   Best composite score: {study.best_value:.4f}")
            save_path = os.path.join(weights_dir, f'best_params_{target_name}.json')
            with open(save_path, 'w') as f:
                json.dump(best_hp, f, indent=2)
            print(f"   Saved to: {save_path}")
        else:
            print(f"\n⚠️ No trial met clinical validity threshold (≥0.60).")

        return best_hp, study

    except Exception as e:
        print(f"❌ Critical error during tuning: {str(e)}")
        traceback.print_exc()
        return None, None

    finally:
        print("\n" + "=" * 80)

In [2]:
TUNING_CONFIG = {
    'epochs': 5000,
    'patience': 100,
    'early_stop_metric': 'f1',
    'use_focal_loss': True,

    'lr_range': (2.31e-5, 2.51e-5),
    'weight_decay_range': (1.12e-6, 2.24e-6),

    'embed_dim': [512],
    'num_heads': [16],
    'num_layers_range': (6, 7),
    'n_kv_heads_options': [4, 8],

    'embed_dim_age': [16, 32],
    'embed_dim_ext': [16],
    'patch_size': [3],
    'patch_embed_ratio': [0.75],
    'temp_attn_hidden': [512],

    'dropout_input_range': (0.1632, 0.1971),
    'dropout_patch_range': (0.1259, 0.2379),
    'dropout_attn_range': (0.1506, 0.1640),
    'dropout_ffn_range': (0.2274, 0.2938),
    'dropout_classifier_range': (0.0338, 0.0779),
    'dropout_temporal_range': (0.1251, 0.1799),

    'stochastic_depth_rate_range': (0.0443, 0.1287),

    'metric_threshold': 0.60,
    'metric_weights': {
        'f1': 0.20,
        'roc_auc': 0.20,
        'accuracy': 0.20,
        'precision': 0.20,
        'recall': 0.20
    },

    'n_startup_trials': 3,
    'n_warmup_steps': 0,
    'interval_steps': 1,
    'base_seed': 42,
}

In [3]:
# For ASD
run_tuning_workflow(
    target_name='ASD',
    data_path='/mnt/movement/users/jaizor/xtra/data/fmri/abide/fmri_ASD.npz',
    pheno_path='/mnt/movement/users/jaizor/xtra/data/fmri/abide/pheno_ASD.csv',
    tuning_config=TUNING_CONFIG,
    n_trials=50
)



🚀 Starting hyperparameter tuning for 'ASD'
📂 Data: /mnt/movement/users/jaizor/xtra/data/fmri/abide/fmri_ASD.npz
   Attempting batch_size = 64...


[I 2026-02-15 01:37:18,888] A new study created in memory with name: bbtransformer_v2_ASD_20260215_013718


Cohort: 585 subjects (271 cases, 314 controls, 46.3% prevalence)
Splits → Train: 409, Val: 88, Test: 88
   ✅ Success with batch_size = 64

✅ Data loaded:
   - Train: 409 | Val: 88
   - Feature dim: 414 | Batch size: 64

🚀 STARTING HYPERPARAMETER TUNING: ASD
Study: bbtransformer_v2_ASD_20260215_013718
Trials: 50
Epochs per trial: 5000
Patience: 100
Metric Threshold: 0.6



  0%|          | 0/50 [00:00<?, ?it/s]


[Trial 0] Selected hyperparameters:
  Architecture: embed_dim=512, num_heads=16, n_kv_heads=8
  Dropout classifier: 0.0767
  Dropout FFN: 0.2526
  Config n_kv_heads_options: [4, 8]
  Valid KV heads: [4, 8]

[Trial 0] Creating model...
[Trial 0] Model created: 30,272,384 parameters
[Trial 0] Starting training (max 5000 epochs)...


...

Early stopping at epoch 288 (F1: 0.2308)
[Trial 48] Training completed
[Trial 48] Evaluating...


[Trial 48] Results:
   F1: 0.6796, ROC-AUC: 0.6323, Acc: 0.6250
   Composite: 0.6710, Valid: False
[I 2026-02-15 05:19:47,349] Trial 48 finished with value: 0.4354838709677419 and parameters: {'embed_dim': 512, 'num_heads': 16, 'n_kv_heads': 8, 'num_layers': 6, 'dropout_input': 0.19033078916680574, 'dropout_patch': 0.1436245608159099, 'dropout_attn': 0.15262111845193405, 'dropout_ffn': 0.253875204666733, 'dropout_classifier': 0.04722172758113339, 'dropout_temporal': 0.15776347670966395, 'embed_dim_age': 32, 'embed_dim_ext': 16, 'patch_size': 3, 'patch_embed_ratio': 0.75, 'temp_attn_hidden': 512, 'stochastic_depth_rate': 0.08952640888790392, 'lr': 2.4141758573564032e-05, 'weight_decay': 1.7325786654654242e-06}. Best is trial 31 with value: -0.753916149173708.

[Trial 49] Selected hyperparameters:
  Architecture: embed_dim=512, num_heads=16, n_kv_heads=4
  Dropout classifier: 0.0341
  Dropout FFN: 0.2358
  Config n_kv_heads_options: [4, 8]
  Valid KV heads: [4, 8]

[Trial 49] Creating mo

Early stopping at epoch 334 (F1: 0.5897)
[Trial 49] Training completed
[Trial 49] Evaluating...


[Trial 49] Results:
   F1: 0.7273, ROC-AUC: 0.6993, Acc: 0.6932
   Composite: 0.7237, Valid: True
[I 2026-02-15 05:26:04,164] Trial 49 finished with value: -0.7236932926422576 and parameters: {'embed_dim': 512, 'num_heads': 16, 'n_kv_heads': 4, 'num_layers': 6, 'dropout_input': 0.17555072192597465, 'dropout_patch': 0.16093578880459602, 'dropout_attn': 0.15772766636188407, 'dropout_ffn': 0.23577277261906732, 'dropout_classifier': 0.03407838279645683, 'dropout_temporal': 0.14301711627450922, 'embed_dim_age': 32, 'embed_dim_ext': 16, 'patch_size': 3, 'patch_embed_ratio': 0.75, 'temp_attn_hidden': 512, 'stochastic_depth_rate': 0.08663050649063378, 'lr': 2.3377561375861744e-05, 'weight_decay': 2.0808928049722896e-06}. Best is trial 31 with value: -0.753916149173708.

✅ CLINICALLY VALID
📊 Trial Statistics:
   Total:     50
   Completed: 50
   Valid:     16 (≥0.60 all metrics)
   Pruned:    0
   Failed:    0

🎯 Best Trial Results:
   Composite Score: 0.7539
   Parameters:      29,354,880
   W

({'embed_dim': 512,
  'num_heads': 16,
  'n_kv_heads': 4,
  'num_layers': 7,
  'dropout_input': 0.18136004006282452,
  'dropout_patch': 0.16501094464177737,
  'dropout_attn': 0.15171971434774525,
  'dropout_ffn': 0.2566453790896773,
  'dropout_classifier': 0.07221857432853679,
  'dropout_temporal': 0.16238543875751235,
  'embed_dim_age': 32,
  'embed_dim_ext': 16,
  'patch_size': 3,
  'patch_embed_ratio': 0.75,
  'temp_attn_hidden': 512,
  'stochastic_depth_rate': 0.07144421481835657,
  'lr': 2.3157039809812987e-05,
  'weight_decay': 1.1402692878411914e-06},
 <optuna.study.study.Study at 0x7f42c0d6a3c0>)